# LAB S06: Diseño de pipelines de extremo a extremo

Cada sección usa el archivo de `mi_pipeline` que se escribe en su paso de la guía; el cuaderno solo importa y ejecuta. La cadena de hoy es control, ingesta, OCR y PLN sobre una bandeja de 200 archivos: se configura con el YAML, resiste los archivos dañados, reparte el trabajo entre procesos y deja un registro de lo que pasó con cada archivo. Los documentos son ficticios y cada uno trae su respuesta correcta para medir. Ningún documento sale de la VM.

## 0. Dónde están los datos de práctica

In [1]:
# Celda estándar de localización de datos (no se modifica)
import os
from pathlib import Path


def localizar_carpeta(nombre="02_Datos_Practica", max_niveles=6, obligatorio=True):
    """Busca `nombre` en la carpeta actual y en sus carpetas superiores."""
    actual = Path.cwd().resolve()
    for carpeta in [actual, *actual.parents][: max_niveles + 1]:
        candidata = carpeta / nombre
        if candidata.is_dir():
            return candidata
    # Respaldo: carpetas del curso en la unidad del sistema (VM de laboratorio)
    curso = Path(os.environ.get("SystemDrive", "C:") + os.sep) / "curso"
    respaldo = [curso / "paquete" / nombre]
    if nombre == "02_Datos_Practica":
        respaldo.insert(0, curso / "datos")
    for candidata in respaldo:
        if candidata.is_dir():
            return candidata
    if obligatorio:
        raise FileNotFoundError(
            f"No se encuentra la carpeta '{nombre}'. El cuaderno se abre desde su carpeta "
            "dentro del repositorio del curso, sin moverlo de lugar."
        )
    return None


DATOS = localizar_carpeta("02_Datos_Practica")
print("Datos de práctica encontrados:", DATOS.name)
print("Subcarpetas:", sorted(p.name for p in DATOS.iterdir() if p.is_dir()))


Datos de práctica encontrados: datos
Subcarpetas: ['01_Lote_S02', '02_Oficios', '03_Estados_Financieros', '04_Contratos', '05_Polizas', '06_Reclamos', '07_Tarifarios', '08_Vouchers', '09_Formularios', '10_Ground_Truth', '11_Salidas_Precalculadas', '12_Actas', '13_Expedientes', '14_Publico_Real']


## Paso 0: conexión con `mi_pipeline`, rama de trabajo y bandeja (C0)

In [2]:
# Conexión de mi_pipeline y de las herramientas del LAB (no se modifica)
%load_ext autoreload
%autoreload 2
import html
import json
import re
import shutil
import subprocess
import sys
import tempfile
import time
import unicodedata
import warnings

import pandas as pd
import pymupdf
import pytesseract
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)   # avisos de cambios futuros de las bibliotecas, que no cambian el resultado
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", None)


def localizar_raiz(max_niveles=6):
    """Carpeta del repositorio clonado que contiene mi_pipeline."""
    if os.environ.get("MI_PIPELINE_RAIZ"):
        return Path(os.environ["MI_PIPELINE_RAIZ"])
    actual = Path.cwd().resolve()
    for carpeta in [actual, *actual.parents][: max_niveles + 1]:
        if (carpeta / "mi_pipeline" / "__init__.py").is_file():
            return carpeta
    clon = Path(os.environ.get("SystemDrive", "C:") + os.sep) / "curso" / "trabajo" / "entorno-extraccion-documental"
    if (clon / "mi_pipeline" / "__init__.py").is_file():
        return clon
    raise FileNotFoundError("No se encuentra la carpeta mi_pipeline: la preparación de la sesión la repone.")


RAIZ = localizar_raiz()
sys.path.insert(0, str(RAIZ))
SALIDAS = Path("salidas")
SALIDAS.mkdir(exist_ok=True)
CONTROLES = []
INICIO = time.perf_counter()
pytesseract.pytesseract.tesseract_cmd = (
    os.environ.get("TESSERACT_CMD")
    or shutil.which("tesseract")
    or str(Path(os.environ.get("ProgramFiles", "")) / "Tesseract-OCR" / "tesseract.exe")
)


def correr_pruebas():
    """Corre las pruebas de mi_pipeline con pytest, muestra el resumen y dice si todas pasaron."""
    corrida = subprocess.run(
        [sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider",
         "--basetemp", tempfile.mkdtemp(prefix="pytest_s06_"), str(RAIZ / "mi_pipeline" / "tests")],
        capture_output=True, text=True, cwd=str(RAIZ), timeout=900, encoding="utf-8", errors="replace",
    )
    resumen = [linea for linea in corrida.stdout.strip().splitlines() if " passed" in linea or " failed" in linea or "error" in linea]
    print(re.sub(r" in [\d.]+s.*", "", resumen[-1]) if resumen else corrida.stdout[-500:])
    return corrida.returncode == 0


# Cómo se muestran los resultados: tablas alineadas, con el texto completo y los números como se escriben en el Perú
PORCENTAJES = {"exactitud"}
SIN_MILES = {"contexto", "nivel"}


def num(valor, decimales=2):
    """Un número con punto de miles, coma decimal y sin ceros de más: 7.728 o 10,78."""
    texto = f"{valor:,.{decimales}f}".replace(",", "§").replace(".", ",").replace("§", ".")
    return texto.rstrip("0").rstrip(",") if "," in texto else texto


def pct(valor):
    """Una proporción como porcentaje: 95,8 %."""
    return f"{valor * 100:.1f} %".replace(".", ",")


def si_no(valor):
    return "sí" if valor else "no"


def como_texto(valor, columna="", vacio="vacío"):
    """Un valor tal como se muestra en una tabla: sí o no, números en formato local y «vacío» si no hay dato."""
    if pd.api.types.is_scalar(valor) and pd.isna(valor):
        return vacio
    if hasattr(valor, "item"):
        valor = valor.item()
    if isinstance(valor, bool):
        return si_no(valor)
    if isinstance(valor, (int, float)):
        if columna in PORCENTAJES:
            return pct(valor)
        if columna in SIN_MILES:
            return str(valor)
        return num(valor, 3 if "CER" in columna else 2)
    return str(valor)


def ver(tabla, titulo, vacio="vacío"):
    """Muestra una tabla alineada y con el texto completo; la tabla original no cambia."""
    if tabla.empty:
        print(f"{titulo}. Ninguno.")
        return
    base = tabla.reset_index() if tabla.index.name else tabla
    vista = pd.DataFrame({columna: [como_texto(v, str(columna), vacio) for v in base[columna]] for columna in base.columns})
    cuerpo = (vista.to_html(index=False, border=0)
              .replace("<th>", '<th style="text-align:left;vertical-align:bottom;padding:4px 10px">')
              .replace("<td>", '<td style="text-align:left;vertical-align:top;white-space:normal;padding:4px 10px">'))
    display({"text/html": f"<p><b>{html.escape(titulo)}</b></p>{cuerpo}", "text/plain": titulo + "\n" + vista.to_string(index=False)}, raw=True)


def decir(texto):
    """Dice qué mostró esta corrida, con sus propios números."""
    display({"text/html": f'<p style="border-left:4px solid #888;padding:6px 10px"><b>En esta corrida:</b> {html.escape(texto)}</p>',
             "text/plain": "En esta corrida: " + texto}, raw=True)


# Medición contra la respuesta correcta de cada documento (la misma vara en todos los ejercicios)
NO_SE_MIDEN = {"ambiguedad", "cuadra", "paginas", "n_paginas", "monto_en_texto", "canal", "adjunto",
               "n_requisitos", "n_requisitos_cumplen", "n_requisitos_no_cumplen", "n_requisitos_observados"}
MESES = ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto", "setiembre", "octubre", "noviembre", "diciembre"]


def llano(texto):
    """Minúsculas, sin tildes y con un solo espacio."""
    base = "".join(c for c in unicodedata.normalize("NFD", str(texto)) if unicodedata.category(c) != "Mn")
    return re.sub(r"\s+", " ", base.lower()).strip(" .,:;")


def figura(valor, texto):
    """Dice si una fecha (en cualquiera de sus formas escritas) o un código figura en el texto."""
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", valor):
        anio, mes, dia = valor.split("-")
        meses = [MESES[int(mes) - 1]] + (["septiembre"] if mes == "09" else [])
        formas = [valor, f"{dia}/{mes}/{anio}", f"{int(dia)}/{mes}/{anio}"] + [f"{d} de {m} de {anio}" for m in meses for d in (dia, int(dia))]
        return any(forma in texto for forma in formas)
    return llano(valor) in texto


def correcto_de(id_):
    """Respuesta correcta de un documento: sus campos, con null en las fechas y códigos que no figuran en su texto."""
    datos = json.loads((DATOS / "10_Ground_Truth" / f"{id_}.json").read_text(encoding="utf-8"))
    texto = llano(datos["texto_completo"])
    correcto = {}
    for campo, valor in datos["campos"].items():
        if campo in NO_SE_MIDEN or isinstance(valor, (bool, dict, list)):
            continue
        if isinstance(valor, str) and (re.fullmatch(r"\d{4}-\d{2}-\d{2}", valor) or campo == "numero_reclamo") and not figura(valor, texto):
            valor = None
        correcto[campo] = valor
    return correcto


def es_correcto(valor, correcto):
    """Medida estricta: el dato listo para usar (número como número, fecha AAAA-MM-DD, moneda PEN); null es correcto si no hay dato."""
    if correcto is None:
        return valor is None
    if valor is None:
        return False
    if isinstance(correcto, (int, float)):
        return isinstance(valor, (int, float)) and not isinstance(valor, bool) and abs(valor - correcto) < 0.011
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}|\d{8,20}", str(correcto)):
        return str(valor).strip() == correcto
    return llano(valor) == llano(correcto)


def aciertos(extraido, correcto):
    """(correctos, esperados, fallos) de un documento."""
    escrito = lambda v: "vacío" if v is None else repr(v)
    fallos = [f"{campo}: {escrito(extraido.get(campo))} en lugar de {escrito(valor)}" for campo, valor in correcto.items() if not es_correcto(extraido.get(campo), valor)]
    return len(correcto) - len(fallos), len(correcto), fallos


# S06: los procesos trabajadores heredan esta variable y así encuentran Tesseract
os.environ["TESSERACT_CMD"] = pytesseract.pytesseract.tesseract_cmd
try:
    import pipeline_sbs  # el paquete del curso: trae el script y el control de ingesta de S02
except ImportError:
    sys.path.insert(0, str(RAIZ / "04_pipeline_sbs"))


def leer_lista():
    """La lista de la bandeja (bandeja_s06.json): ID, carpeta, archivo y tipo de cada documento."""
    for candidata in (RAIZ / "mi_pipeline" / "config" / "bandeja_s06.json", RAIZ / "05_Laboratorios" / "datos_s06" / "bandeja_s06.json"):
        if candidata.is_file():
            return json.loads(candidata.read_text(encoding="utf-8"))
    raise FileNotFoundError("No se encuentra bandeja_s06.json: la preparación de la sesión la trae del repositorio.")


def armar_bandeja(lista="completa"):
    """Arma la bandeja de la Mesa de Partes DEMO junto al repositorio: copia los archivos de la lista desde los datos de práctica."""
    bandeja = leer_lista()
    elegidos = [fila for fila in bandeja["completa"] if lista == "completa" or fila["id"] in set(bandeja["corta"])]
    destino = RAIZ.parent / ("bandeja_s06" if lista == "completa" else "bandeja_s06_corta")
    if not destino.is_dir() or len(list(destino.iterdir())) != len(elegidos):
        shutil.rmtree(destino, ignore_errors=True)
        destino.mkdir(parents=True)
        for fila in elegidos:
            shutil.copy2(DATOS / fila["carpeta"] / fila["archivo"], destino / fila["archivo"])
    return destino


TIPO_DE = {fila["archivo"]: fila["tipo"] for fila in leer_lista()["completa"]}   # tipo de documento de cada archivo de la bandeja

print("Carpeta del repositorio:", RAIZ.name, "| Tesseract:", pytesseract.get_tesseract_version())

Carpeta del repositorio: entorno-extraccion-documental | Tesseract: 5.4.0.20240606


In [3]:
# ✔ Control C0: la rama de trabajo y la bandeja de la Mesa de Partes DEMO
LISTA = "completa"   # "completa" = 200 archivos; "corta" = 60 archivos (solo si el docente lo indica porque la VM va lenta)
ESPERADO = {"completa": {"archivos": 200, "ok": 194, "campos": 2059}, "corta": {"archivos": 60, "ok": 54, "campos": 576}}[LISTA]
rama = subprocess.run(["git", "branch", "--show-current"], cwd=RAIZ, capture_output=True, text=True).stdout.strip()
assert rama == "trabajo-s06", f"C0: la rama actual es '{rama}'; la preparación de la sesión crea trabajo-s06 desde trabajo-s05."
BANDEJA = armar_bandeja(LISTA)
archivos = sorted(BANDEJA.iterdir())
formatos = pd.Series([ruta.suffix.lower() for ruta in archivos]).value_counts().rename_axis("formato").rename("archivos").reset_index()
ver(formatos, f"La bandeja: {len(archivos)} archivos en la carpeta {BANDEJA.name}")
assert len(archivos) == ESPERADO["archivos"], f"C0: la bandeja tiene {len(archivos)} archivos y se esperaban {ESPERADO['archivos']}."
CONTROLES.append("C0")
print("✔ C0 superado: rama", rama, "| bandeja de", len(archivos), "archivos")

formato,archivos
.pdf,116
.eml,33
.docx,21
.jpg,12
.html,10
.png,8


✔ C0 superado: rama trabajo-s06 | bandeja de 200 archivos


**Cómo leer esta salida**

- **Qué se ve:** Cuántos archivos de cada formato tiene la bandeja.
- **Qué mirar:** Hay seis formatos distintos: cada uno necesita su lector, y las imágenes y los PDF escaneados necesitan OCR.
- **Qué se decide:** Nada todavía: es el inventario de lo que llegó.

## Reto: ¿cuánto demora el script de S02 con 200 documentos? (C0r)

In [4]:
# Reto: ¿cuánto demora el script de S02 con los 200 documentos? (C0r)
APUESTA_S = 60   # la apuesta, en segundos: se cambia este número antes de ejecutar la celda
from pipeline_sbs.orquestacion.pipeline import ejecutar_pipeline

inicio = time.perf_counter()
tabla_s02 = ejecutar_pipeline(BANDEJA, {"cuarentena": str(SALIDAS / "cuarentena_s02")})
segundos_s02 = time.perf_counter() - inicio
estados_s02 = tabla_s02["estado"].value_counts()
ver(estados_s02.rename_axis("estado").rename("archivos").reset_index(), "El script de S02 sobre la bandeja")
ver(tabla_s02[tabla_s02["estado"] != "ok"][["archivo", "estado", "motivo"]], "Lo que el script de S02 no dejó pasar")
decir(f"el script de S02 revisó {len(tabla_s02)} archivos en {num(segundos_s02, 1)} s (la apuesta fue {APUESTA_S} s). "
      f"Marcó {int(tabla_s02['requiere_ocr'].sum())} que necesitan OCR, pero no los leyó ni extrajo un solo campo.")
assert (int(estados_s02.get("ok", 0)), int(estados_s02.get("fallo", 0)), int(estados_s02.get("duplicado", 0))) == (ESPERADO["ok"], 5, 1), \
    "C0r: se esperaban los archivos aceptados, 5 fallos y 1 duplicado."
CONTROLES.append("C0r")
print("✔ C0r superado:", ESPERADO["ok"], "ok, 5 fallos y 1 duplicado")

estado,archivos
ok,194
fallo,5
duplicado,1


archivo,estado,motivo
LOT-0002.pdf,fallo,extension_falsa
LOT-0006.docx,fallo,docx_corrupto
LOT-0007.pdf,fallo,pdf_cifrado
LOT-0014.pdf,fallo,pdf_danado
LOT-0016.pdf,fallo,archivo_vacio
LOT-0019.pdf,duplicado,duplicado


En esta corrida: el script de S02 revisó 200 archivos en 3,7 s (la apuesta fue 60 s). Marcó 70 que necesitan OCR, pero no los leyó ni extrajo un solo campo.

✔ C0r superado: 194 ok, 5 fallos y 1 duplicado


**Cómo leer esta salida**

- **Qué es cada fila:** En la primera tabla, un estado y cuántos archivos lo tienen; en la segunda, cada archivo que no pasó, con su motivo.
- **Qué mirar:** El tiempo total frente a la apuesta, y cuántos archivos quedaron marcados para OCR sin ser leídos.
- **Qué se decide:** El script de S02 sirve para recibir y controlar; no entrega ningún dato del documento.

In [5]:
# Reto: la cadena completa (ingesta, OCR y PLN) sobre 10 documentos de la misma bandeja, y la proyección para todos
from mi_pipeline.extraccion.hibrido import pln
from mi_pipeline.ingesta.leer_documento import aplicar_ocr, campos_rotulados, leer_documento
from mi_pipeline.llm.esquemas import campos_de

MUESTRA_RETO = ['POL-0001.pdf', 'CON-0001.docx', 'CON-0021.pdf', 'OFI-0001.pdf', 'OFI-0002.pdf', 'EFI-0001.pdf', 'EFI-0031.pdf', 'REC-0001.eml', 'VOU-0001.jpg', 'TAR-0001.html']
filas = []
for nombre in MUESTRA_RETO:
    tipo, t0 = TIPO_DE[nombre], time.perf_counter()
    doc = leer_documento(BANDEJA / nombre)                                   # ingesta: el lector del formato
    campos = {campo: campos_rotulados(tipo, doc).get(campo) for campo in campos_de(tipo)}
    t1 = time.perf_counter()
    if doc["pendientes"]:                                                     # OCR: solo lo que no trae capa de texto
        doc = aplicar_ocr(doc, motor="tesseract")
        campos = {campo: campos_rotulados(tipo, doc).get(campo) for campo in campos_de(tipo)}
    t2 = time.perf_counter()
    campos = pln(tipo, doc, campos)                                           # PLN: reglas y NER
    t3 = time.perf_counter()
    filas.append({"archivo": nombre, "ingesta (s)": t1 - t0, "OCR (s)": t2 - t1, "PLN (s)": t3 - t2, "total (s)": t3 - t0,
                  "campos llenos": sum(valor is not None for valor in campos.values()), "campos": len(campos)})
tabla_reto = pd.DataFrame(filas)
ver(tabla_reto.round(2), "La cadena completa sobre 10 documentos")
segundos_10 = float(tabla_reto["total (s)"].sum())
proyeccion_s = segundos_10 / len(MUESTRA_RETO) * ESPERADO["ok"]
por_etapa_reto = tabla_reto[["ingesta (s)", "OCR (s)", "PLN (s)"]].sum()
decir(f"10 documentos tardaron {num(segundos_10, 1)} s; a ese ritmo, los {ESPERADO['ok']} tardan unos {num(proyeccion_s / 60, 1)} minutos "
      f"({num(proyeccion_s / max(segundos_s02, 0.1), 0)} veces el script de S02). La etapa que más tarda es {por_etapa_reto.idxmax().replace(' (s)', '')}: "
      f"{pct(por_etapa_reto.max() / por_etapa_reto.sum())} del tiempo.")

archivo,ingesta (s),OCR (s),PLN (s),total (s),campos llenos,campos
POL-0001.pdf,"0,01",0,0,"0,01",13,13
CON-0001.docx,"0,05",0,"1,47","1,52",7,15
CON-0021.pdf,"0,04",0,"0,37","0,41",7,15
OFI-0001.pdf,0,"2,78",0,"2,79",7,10
OFI-0002.pdf,0,"3,44",0,"3,44",7,10
EFI-0001.pdf,"0,01",0,0,"0,01",8,9
EFI-0031.pdf,0,"7,2",0,"7,21",4,9
REC-0001.eml,0,"0,47",0,"0,47",8,10
VOU-0001.jpg,0,"0,36",0,"0,36",2,10
TAR-0001.html,0,0,0,0,3,4


En esta corrida: 10 documentos tardaron 16,2 s; a ese ritmo, los 194 tardan unos 5,2 minutos (86 veces el script de S02). La etapa que más tarda es OCR: 87,9 % del tiempo.

**Cómo leer esta salida**

- **Qué es cada fila:** Un documento, con los segundos de cada etapa y cuántos campos quedaron llenos.
- **Qué mirar:** La columna `OCR (s)`: es cero en los documentos digitales y la mayor en los escaneados.
- **Qué se decide:** Si la bandeja completa tarda minutos en un cuaderno, hace falta un pipeline que se configure, resista fallos y reparta el trabajo.

## E1. Etapas encadenadas desde el YAML (C1 y C2)

◆ **Diseño de pipelines.** Un pipeline es una fila de etapas con un contrato de datos entre ellas. Lo que cambia va al YAML, no al código. Un YAML validado detecta el error antes de procesar.

In [7]:
# E1: el YAML validado. El archivo del paquete carga; una clave mal escrita se rechaza antes de procesar (C1)
from mi_pipeline.orquestacion.config import RUTA_CONFIG, ErrorConfiguracion, cargar_config

CFG = cargar_config().en_carpeta(SALIDAS)
ver(pd.DataFrame([{"clave": f"orquestacion.{clave}", "valor": str(valor)} for clave, valor in CFG.orquestacion.model_dump().items() if clave != "tipo_por_prefijo"]
                 + [{"clave": "reintentos.espera_s", "valor": str(CFG.reintentos.espera_s)}, {"clave": "almacenamiento.duckdb", "valor": CFG.almacenamiento.duckdb}]),
    "Lo que dice el YAML sobre la ejecución")

con_error = SALIDAS / "pipeline_con_error.yaml"
con_error.write_text(RUTA_CONFIG.read_text(encoding="utf-8").replace("  trabajadores: 4", "  trabajdores: 8"), encoding="utf-8")
try:
    cargar_config(con_error)
    mensaje_c1 = ""
except ErrorConfiguracion as error:
    mensaje_c1 = str(error)
print("Con la clave mal escrita:\n" + mensaje_c1)
assert CFG.orquestacion.etapas == ["control", "ingesta", "ocr", "pln"], "C1: el YAML debe declarar las cuatro etapas en orden."
assert "trabajdores" in mensaje_c1, "C1: la clave mal escrita debe rechazarse con su mensaje."
CONTROLES.append("C1")
print("✔ C1 superado: el YAML válido carga y la clave mal escrita se rechaza")

clave,valor
orquestacion.modo,procesos
orquestacion.trabajadores,4
orquestacion.etapas,"['control', 'ingesta', 'ocr', 'pln']"
orquestacion.motor_ocr,tesseract
reintentos.espera_s,"[1.0, 2.0, 4.0]"
almacenamiento.duckdb,salidas\resultados.duckdb


Con la clave mal escrita:
pipeline_con_error.yaml no es válido:
orquestacion.trabajdores: clave desconocida. ¿Se quiso escribir 'trabajadores'?
✔ C1 superado: el YAML válido carga y la clave mal escrita se rechaza


**Cómo leer esta salida**

- **Qué se ve:** Las claves del YAML que gobiernan la ejecución y, debajo, el mensaje que da una clave mal escrita.
- **Qué mirar:** El mensaje nombra la clave (`orquestacion.trabajdores`) y sugiere la correcta.
- **Qué se decide:** Un supervisor exige que la configuración se valide antes de procesar: un error de escritura no debe cambiar el resultado en silencio.

In [8]:
# E1: 12 documentos recorren las etapas del YAML: control -> ingesta -> ocr -> pln (C2)
from mi_pipeline.orquestacion.etapas import ETAPAS, ejecutar_etapas

ver(pd.DataFrame([{"etapa": etapa.nombre, "recibe": etapa.recibe, "entrega": etapa.entrega} for etapa in ETAPAS.values()]), "El contrato de cada etapa")
MUESTRA_E1 = ['POL-0002.pdf', 'CON-0002.docx', 'CON-0022.pdf', 'OFI-0003.pdf', 'OFI-0004.pdf', 'EFI-0002.pdf', 'EFI-0032.pdf', 'REC-0002.eml', 'REC-0003.eml', 'TAR-0002.html', 'VOU-0002.jpg', 'VOU-0003.png']
RESULTADOS_E1 = [ejecutar_etapas(BANDEJA / nombre, TIPO_DE[nombre], CFG) for nombre in MUESTRA_E1]
ver(pd.DataFrame([{"archivo": r["archivo"], "tipo": r["tipo"], "estado": r["estado"], **{f"{etapa} (s)": round(r["segundos"][etapa], 2) for etapa in CFG.orquestacion.etapas},
                   "páginas con OCR": r["paginas_ocr"], "campos llenos": len(r["campos"]) - len(r["revisar"]), "a revisar": len(r["revisar"])} for r in RESULTADOS_E1]),
    "12 documentos por las cuatro etapas")
assert all(r["estado"] == "ok" and list(r["fases"]) == ["ingesta", "ocr", "pln"] and set(r["segundos"]) == set(CFG.orquestacion.etapas) for r in RESULTADOS_E1), \
    "C2: los 12 documentos deben recorrer las cuatro etapas y cada etapa debe entregar su parte."
CONTROLES.append("C2")
print("✔ C2 superado: 12 documentos por control, ingesta, ocr y pln")

etapa,recibe,entrega
control,la ruta del archivo,"sha256, formato, estado (ok o cuarentena) y motivo"
ingesta,la ruta de un archivo que pasó el control,el documento leído (partes y pendientes de OCR) y los campos rotulados
ocr,el documento con partes pendientes,el documento con todo su texto y los campos rotulados que faltaban
pln,el documento con su texto y los campos,"los campos completados por reglas y NER, y la lista de ambiguos"


archivo,tipo,estado,control (s),ingesta (s),ocr (s),pln (s),páginas con OCR,campos llenos,a revisar
POL-0002.pdf,poliza,ok,0,0,0,0,0,12,1
CON-0002.docx,contrato,ok,0,"0,04",0,"0,36",0,7,8
CON-0022.pdf,contrato,ok,0,"0,01",0,"0,34",0,7,8
OFI-0003.pdf,oficio,ok,0,0,"3,19",0,1,7,3
OFI-0004.pdf,oficio,ok,0,0,"3,51",0,1,7,3
EFI-0002.pdf,estado_financiero,ok,0,0,0,0,0,8,1
EFI-0032.pdf,estado_financiero,ok,0,0,"4,69",0,2,8,1
REC-0002.eml,reclamo,ok,0,0,0,0,0,9,1
REC-0003.eml,reclamo,ok,0,"0,01","0,42",0,1,7,3
TAR-0002.html,tarifario,ok,0,0,0,0,0,3,1


✔ C2 superado: 12 documentos por control, ingesta, ocr y pln


**Cómo leer esta salida**

- **Qué es cada fila:** En la primera tabla, una etapa con lo que recibe y lo que entrega; en la segunda, un documento con los segundos de cada etapa.
- **Qué mirar:** Qué documentos gastan tiempo en `ocr (s)` y cuántos campos deja cada uno «a revisar».
- **Qué se decide:** Si una etapa cambia por dentro pero entrega lo mismo, las demás no se tocan.

In [9]:
# E1: configurable. Se quita la etapa ocr de la configuración y el mismo oficio escaneado se procesa de nuevo
sin_ocr = CFG.con(etapas=["control", "ingesta", "pln"])
completo, recortado = ejecutar_etapas(BANDEJA / "OFI-0003.pdf", "oficio", CFG), ejecutar_etapas(BANDEJA / "OFI-0003.pdf", "oficio", sin_ocr)
ver(pd.DataFrame([{"etapas": " -> ".join(cfg.orquestacion.etapas), "segundos": round(sum(r["segundos"].values()), 2), "caracteres leídos": r["caracteres"],
                   "campos llenos": len(r["campos"]) - len(r["revisar"]), "a revisar": len(r["revisar"])} for cfg, r in ((CFG, completo), (sin_ocr, recortado))]),
    "El mismo oficio con y sin la etapa ocr")
decir(f"sin la etapa ocr, el oficio escaneado deja {len(recortado['revisar'])} campos para revisar en lugar de {len(completo['revisar'])}: "
      "la etapa se quitó en la configuración, sin tocar el código.")

etapas,segundos,caracteres leídos,campos llenos,a revisar
control -> ingesta -> ocr -> pln,"3,16",1.074,7,3
control -> ingesta -> pln,0,0,0,10


En esta corrida: sin la etapa ocr, el oficio escaneado deja 10 campos para revisar en lugar de 3: la etapa se quitó en la configuración, sin tocar el código.

**Cómo leer esta salida**

- **Qué es cada fila:** El mismo oficio escaneado con las cuatro etapas y sin la etapa `ocr`.
- **Qué mirar:** Los caracteres leídos y los campos a revisar de cada fila.
- **Qué se decide:** Qué etapas corren se decide en la configuración; quitar una tiene un costo que se ve en los datos.

## E2. Idempotencia, reintentos y cuarentena (C3 y C4)

In [10]:
# E2: el registro. Los 7 archivos del lote de S02: 1 pasa, 5 van a cuarentena con su motivo y 1 es un duplicado (C3)
from mi_pipeline.orquestacion.registro import a_cuarentena, abrir_registro, huella, marcar, reintentar, ya_procesado

shutil.rmtree(SALIDAS / "e2", ignore_errors=True)   # la celda parte de cero cada vez que se ejecuta
CFG_E2 = cargar_config().en_carpeta(SALIDAS / "e2")
LOTE_E2 = sorted(ruta for ruta in BANDEJA.iterdir() if ruta.name.startswith("LOT-"))


def corrida_e2():
    """Una corrida del control sobre los 7 archivos: salta lo ya registrado, aparta lo dañado y marca el duplicado."""
    con, vistos, filas = abrir_registro(CFG_E2.almacenamiento.duckdb), {}, []
    huellas = {ruta: huella(ruta) for ruta in LOTE_E2}
    ya_hechos = {sha256 for sha256 in huellas.values() if ya_procesado(con, sha256)}    # lo que el registro traía antes de esta corrida
    for ruta, sha256 in huellas.items():
        if sha256 in ya_hechos:
            filas.append({"archivo": ruta.name, "qué pasó": "saltado: ya está en el registro", "motivo": ""})
            continue
        d = ETAPAS["control"].funcion({"ruta": str(ruta)}, CFG_E2)
        if d["estado"] == "cuarentena":
            a_cuarentena(ruta, CFG_E2.cuarentena.carpeta, d["motivo"], d["detalle"])
        elif sha256 in vistos:
            d.update(estado="duplicado", motivo=f"mismo contenido que {vistos[sha256]}")
        else:
            vistos[sha256] = ruta.name
        marcar(con, sha256, ruta.name, d["estado"], d["motivo"])
        filas.append({"archivo": ruta.name, "qué pasó": d["estado"], "motivo": d["motivo"]})
    con.close()
    return pd.DataFrame(filas)


primera_e2 = corrida_e2()
ver(primera_e2, "Primera corrida: el estado de cada archivo")
en_cuarentena = sorted(ruta.name for ruta in Path(CFG_E2.cuarentena.carpeta).iterdir() if ruta.suffix != ".jsonl")
print("Carpeta de cuarentena:", ", ".join(en_cuarentena))
conteo_e2 = primera_e2["qué pasó"].value_counts()
assert (int(conteo_e2.get("cuarentena", 0)), int(conteo_e2.get("duplicado", 0)), int(conteo_e2.get("ok", 0))) == (5, 1, 1), "C3: se esperaban 5 en cuarentena, 1 duplicado y 1 ok."
assert primera_e2[primera_e2["qué pasó"] == "cuarentena"]["motivo"].nunique() == 5, "C3: cada archivo en cuarentena tiene su propio motivo."
CONTROLES.append("C3")
print("✔ C3 superado: 5 en cuarentena con su motivo y 1 duplicado")

archivo,qué pasó,motivo
LOT-0001.pdf,ok,
LOT-0002.pdf,cuarentena,extension_falsa
LOT-0006.docx,cuarentena,docx_corrupto
LOT-0007.pdf,cuarentena,pdf_cifrado
LOT-0014.pdf,cuarentena,pdf_danado
LOT-0016.pdf,cuarentena,archivo_vacio
LOT-0019.pdf,duplicado,mismo contenido que LOT-0001.pdf


Carpeta de cuarentena: LOT-0002.pdf, LOT-0006.docx, LOT-0007.pdf, LOT-0014.pdf, LOT-0016.pdf
✔ C3 superado: 5 en cuarentena con su motivo y 1 duplicado


**Cómo leer esta salida**

- **Qué es cada fila:** Un archivo del lote con lo que le pasó y su motivo.
- **Qué mirar:** Cinco archivos en cuarentena, cada uno con un motivo distinto, y un duplicado que dice de quién es copia.
- **Qué se decide:** Con el motivo a la vista, una persona decide: pedir el archivo de nuevo, pedir la contraseña o descartarlo.

In [11]:
# E2: idempotencia y reintentos. La segunda corrida no repite nada; un archivo bloqueado se reintenta con 1, 2 y 4 s (C4)
segunda_e2 = corrida_e2()
ver(segunda_e2, "Segunda corrida sobre los mismos archivos")

esperas, llamadas = [], []


def lectura_bloqueada():
    """Simula un archivo bloqueado por otro programa: falla tres veces y a la cuarta se deja leer."""
    llamadas.append(1)
    if len(llamadas) <= 3:
        raise PermissionError("el archivo está en uso por otro programa")
    return "leído"


valor, llamadas_hechas = reintentar(lectura_bloqueada, CFG_E2.reintentos.intentos, CFG_E2.reintentos.espera_s, dormir=esperas.append)


def lectura_danada():
    raise ValueError("el PDF está dañado")


try:
    reintentar(lectura_danada, dormir=esperas.append)
    sin_reintento = False
except ValueError:
    sin_reintento = True
decir(f"la segunda corrida saltó {int((segunda_e2['qué pasó'].str.startswith('saltado')).sum())} de {len(segunda_e2)} archivos. El archivo bloqueado se leyó "
      f"a la llamada {llamadas_hechas}, después de esperar {esperas} segundos (en esta celda la espera se anota y no se cumple, para no detener la clase). "
      "El archivo dañado no se reintentó: va directo a cuarentena.")
assert segunda_e2["qué pasó"].str.startswith("saltado").all(), "C4: la segunda corrida no debe procesar ningún archivo ya registrado."
assert (valor, llamadas_hechas, esperas) == ("leído", 4, [1, 2, 4]) and sin_reintento, "C4: el error transitorio se reintenta con 1, 2 y 4 s; el dañado no."
CONTROLES.append("C4")
print("✔ C4 superado: segunda corrida sin reproceso y reintentos con 1, 2 y 4 s")

archivo,qué pasó,motivo
LOT-0001.pdf,saltado: ya está en el registro,
LOT-0002.pdf,saltado: ya está en el registro,
LOT-0006.docx,saltado: ya está en el registro,
LOT-0007.pdf,saltado: ya está en el registro,
LOT-0014.pdf,saltado: ya está en el registro,
LOT-0016.pdf,saltado: ya está en el registro,
LOT-0019.pdf,saltado: ya está en el registro,


En esta corrida: la segunda corrida saltó 7 de 7 archivos. El archivo bloqueado se leyó a la llamada 4, después de esperar [1.0, 2.0, 4.0] segundos (en esta celda la espera se anota y no se cumple, para no detener la clase). El archivo dañado no se reintentó: va directo a cuarentena.

✔ C4 superado: segunda corrida sin reproceso y reintentos con 1, 2 y 4 s


**Cómo leer esta salida**

- **Qué se ve:** La segunda corrida sobre los mismos archivos y la frase con los reintentos.
- **Qué mirar:** Todos los archivos dicen «saltado»; las esperas del archivo bloqueado son 1, 2 y 4 segundos.
- **Qué se decide:** Se reintenta solo lo que puede arreglarse con esperar; lo dañado va a cuarentena a la primera.

## E3. Reparto del trabajo y cola de escritura; documentos por minuto (C5 y C6)

◆ **Escalabilidad.** Se mide en documentos por minuto y se busca el cuello de botella. Procesos para el trabajo pesado y una cola con un solo escritor para guardar. 4 procesadores lógicos no dan 4 veces: son 2 núcleos físicos.

In [12]:
# E3: la función de trabajo vive en un módulo de mi_pipeline; así los procesos la encuentran
from mi_pipeline.orquestacion.trabajo import procesar_archivo

uno = procesar_archivo(BANDEJA / "VOU-0001.jpg", "voucher", CFG)
ver(pd.DataFrame([{"archivo": uno["archivo"], "estado": uno["estado"], "intentos": uno["intentos"], "segundos": uno["segundos_total"],
                   "campos llenos": len(uno["campos"]) - len(uno["revisar"]), "a revisar": len(uno["revisar"]), "proceso": uno["proceso"]}]),
    "Un archivo procesado por la función de trabajo")
print("Número de este proceso (el del cuaderno):", os.getpid(), "| procesadores lógicos de la VM:", os.cpu_count())

archivo,estado,intentos,segundos,campos llenos,a revisar,proceso
VOU-0001.jpg,ok,1,"0,38",2,8,11.712


Número de este proceso (el del cuaderno): 11712 | procesadores lógicos de la VM: 4


**Cómo leer esta salida**

- **Qué se ve:** El resultado de un archivo: estado, intentos, segundos y el número del proceso que lo hizo.
- **Qué mirar:** El número de proceso coincide con el del cuaderno; en las corridas con procesos será otro.
- **Qué se decide:** La función de trabajo vive en un módulo para poder repartirla entre procesos.

In [14]:
# E3: la bandeja completa en serie: un archivo tras otro (tarda unos minutos; el avance se imprime cada 25 archivos)
from mi_pipeline.orquestacion.lote import ejecutar_lote


def corrida(modo, trabajadores):
    """Una corrida limpia del lote: borra su carpeta de salidas para medir desde cero."""
    salida = SALIDAS / f"e3_{modo}_{trabajadores}"
    shutil.rmtree(salida, ignore_errors=True)
    cfg = cargar_config().en_carpeta(salida).con(modo=modo, trabajadores=trabajadores)
    return ejecutar_lote(BANDEJA, cfg, avisar=print), cfg


CORRIDAS = {}
CORRIDAS["secuencial"], _ = corrida("secuencial", 1)

Archivos: 200 | ya procesados: 0 | por procesar: 200
Control: 194 aceptados | 6 apartados (cuarentena o duplicado)
  25 de 194 | esperan en la cola: 1
  50 de 194 | esperan en la cola: 11
  75 de 194 | esperan en la cola: 1
  100 de 194 | esperan en la cola: 1
  125 de 194 | esperan en la cola: 7
  150 de 194 | esperan en la cola: 1
  175 de 194 | esperan en la cola: 1
  194 de 194 | esperan en la cola: 1
Listo en 207.1 s | 57.9 documentos por minuto | escritos por la cola: 200


**Cómo leer esta salida**

- **Qué se ve:** El avance de la corrida en serie y, al final, los segundos y los documentos por minuto.
- **Qué mirar:** La línea «esperan en la cola»: en serie casi nunca hay espera.
- **Qué se decide:** Este tiempo es la línea de base: todo lo demás se compara con él.

In [15]:
# E3: la misma bandeja repartida entre 2 procesos
CORRIDAS["2 procesos"], _ = corrida("procesos", 2)

Archivos: 200 | ya procesados: 0 | por procesar: 200
Control: 194 aceptados | 6 apartados (cuarentena o duplicado)
  25 de 194 | esperan en la cola: 1
  50 de 194 | esperan en la cola: 7
  75 de 194 | esperan en la cola: 1
  100 de 194 | esperan en la cola: 1
  125 de 194 | esperan en la cola: 6
  150 de 194 | esperan en la cola: 1
  175 de 194 | esperan en la cola: 1
  194 de 194 | esperan en la cola: 1
Listo en 119.0 s | 100.8 documentos por minuto | escritos por la cola: 200


In [16]:
# E3: la misma bandeja repartida entre 4 procesos
CORRIDAS["4 procesos"], CFG_4 = corrida("procesos", 4)

Archivos: 200 | ya procesados: 0 | por procesar: 200
Control: 194 aceptados | 6 apartados (cuarentena o duplicado)
  25 de 194 | esperan en la cola: 1
  50 de 194 | esperan en la cola: 10
  75 de 194 | esperan en la cola: 1
  100 de 194 | esperan en la cola: 1
  125 de 194 | esperan en la cola: 4
  150 de 194 | esperan en la cola: 1
  175 de 194 | esperan en la cola: 1
  194 de 194 | esperan en la cola: 1
Listo en 94.0 s | 127.7 documentos por minuto | escritos por la cola: 200


In [17]:
# E3: secuencial frente a procesos: mismos resultados, segundos, documentos por minuto y aceleración (C5 y C6)
por_minuto = lambda r: (r.archivos - r.saltados) / r.segundos * 60          # documentos por minuto, sin redondear
base_dpm = por_minuto(CORRIDAS["secuencial"])
tabla_e3 = pd.DataFrame([{"corrida": nombre, "trabajadores": r.trabajadores, "segundos": round(r.segundos, 1), "documentos por minuto": r.documentos_por_minuto,
                          "aceleración": round(por_minuto(r) / base_dpm, 2), "escritos por la cola": r.escritos, "máximo en espera": r.max_en_cola}
                         for nombre, r in CORRIDAS.items()])
ver(tabla_e3, "Secuencial frente a procesos sobre la misma bandeja")
serie = CORRIDAS["secuencial"]
etapas_e3 = pd.DataFrame([{"etapa": etapa, "segundos": segundos, "exactitud": segundos / sum(serie.segundos_por_etapa.values())}
                          for etapa, segundos in serie.segundos_por_etapa.items()]).rename(columns={"exactitud": "parte del tiempo"})
etapas_e3["parte del tiempo"] = etapas_e3["parte del tiempo"].map(pct)
ver(etapas_e3, "Dónde se va el tiempo (corrida en serie)")
estados_e3 = pd.DataFrame([{"corrida": nombre, "ok": r.ok, "cuarentena": r.cuarentena, "duplicados": r.duplicados, "errores": r.errores} for nombre, r in CORRIDAS.items()])
ver(estados_e3, "Estados de cada corrida")
campos_de_corrida = lambda r: {x["archivo"]: x.get("campos") for x in r.resultados}
iguales = all(campos_de_corrida(r) == campos_de_corrida(serie) for r in CORRIDAS.values())
cuello = max(serie.segundos_por_etapa, key=serie.segundos_por_etapa.get)
decir(f"en serie, {num(serie.segundos, 1)} s ({num(serie.documentos_por_minuto, 1)} documentos por minuto); con 4 procesos, {num(CORRIDAS['4 procesos'].segundos, 1)} s "
      f"({num(CORRIDAS['4 procesos'].documentos_por_minuto, 1)} por minuto): {num(por_minuto(CORRIDAS['4 procesos']) / base_dpm, 2)} veces más rápido, no 4. "
      f"El cuello de botella es la etapa {cuello}. Las tres corridas extrajeron los mismos campos: {si_no(iguales)}.")
assert iguales, "C5: secuencial y procesos deben dar los mismos campos."
assert all((r.ok, r.cuarentena, r.duplicados, r.errores) == (ESPERADO["ok"], 5, 1, 0) for r in CORRIDAS.values()), "C5: cada corrida debe dejar los mismos estados."
CONTROLES.append("C5")
assert len(tabla_e3) == 3 and all(r.escritos == ESPERADO["archivos"] for r in CORRIDAS.values()), "C6: la cola debe escribir un resultado por archivo en cada corrida."
CONTROLES.append("C6")
print("✔ C5 y C6 superados: mismos resultados, tabla de rendimiento y un solo escritor")

corrida,trabajadores,segundos,documentos por minuto,aceleración,escritos por la cola,máximo en espera
secuencial,1,"207,1","57,9",1,200,21
2 procesos,2,119,"100,8","1,74",200,16
4 procesos,4,94,"127,7","2,2",200,19


etapa,segundos,parte del tiempo
control,"0,3","0,1 %"
ingesta,"1,5","0,7 %"
ocr,"192,7","93,2 %"
pln,"12,3","5,9 %"


corrida,ok,cuarentena,duplicados,errores
secuencial,194,5,1,0
2 procesos,194,5,1,0
4 procesos,194,5,1,0


En esta corrida: en serie, 207,1 s (57,9 documentos por minuto); con 4 procesos, 94 s (127,7 por minuto): 2,2 veces más rápido, no 4. El cuello de botella es la etapa ocr. Las tres corridas extrajeron los mismos campos: sí.

✔ C5 y C6 superados: mismos resultados, tabla de rendimiento y un solo escritor


**Cómo leer esta salida**

- **Qué es cada fila:** Una corrida de la misma bandeja: en serie, con 2 y con 4 procesos.
- **Qué dice cada columna:** `documentos por minuto`: archivos atendidos por minuto. `aceleración`: cuántas veces más rápido que en serie. `escritos por la cola`: resultados que anotó el único escritor. `máximo en espera`: el mayor número de resultados que esperaron a la vez.
- **Qué mirar:** La aceleración con 4 procesos queda lejos de 4, y una sola etapa se lleva casi todo el tiempo.
- **Qué se decide:** Cuántos trabajadores conviene usar en esta VM y qué etapa hay que mejorar primero.

## E4. Almacenamiento, reporte y línea de comandos (C7 y C8)

◆ **Orquestación y automatización.** Un comando con su configuración reemplaza al cuaderno. El registro dice qué se procesó, qué falló y por qué. Aquí basta un ejecutor propio y el programador de tareas de Windows.

In [18]:
# E4: almacenamiento. Los resultados de la corrida con 4 procesos van a DuckDB y a Parquet; repetir no cambia las filas (C7)
from mi_pipeline.almacenamiento.guardar import exportar_parquet, guardar_resultados, leer_tabla

filas_1 = guardar_resultados(CFG_4.almacenamiento.duckdb, CORRIDAS["4 procesos"].resultados)
parquet_1 = exportar_parquet(CFG_4.almacenamiento.duckdb, CFG_4.almacenamiento.parquet)
SEGUNDA = ejecutar_lote(BANDEJA, CFG_4, avisar=print)      # la misma bandeja, otra vez, sobre la misma base
filas_2 = guardar_resultados(CFG_4.almacenamiento.duckdb, SEGUNDA.resultados)
parquet_2 = exportar_parquet(CFG_4.almacenamiento.duckdb, CFG_4.almacenamiento.parquet)
ver(pd.DataFrame([{"tabla": tabla, "filas en DuckDB": filas_1[tabla], "filas en Parquet": parquet_1[tabla],
                   "DuckDB tras repetir": filas_2[tabla], "Parquet tras repetir": parquet_2[tabla]} for tabla in ("documentos", "campos", "registro")]),
    "Filas de cada tabla, antes y después de repetir la corrida")
ver(leer_tabla(CFG_4.almacenamiento.duckdb, "registro").groupby("estado").size().rename("archivos").reset_index(), "El registro: archivos por estado")
decir(f"la segunda corrida saltó {SEGUNDA.saltados} archivos y procesó {SEGUNDA.archivos - SEGUNDA.saltados}, en {num(SEGUNDA.segundos, 1)} s. "
      f"La base conserva {filas_2['documentos']} documentos, {num(filas_2['campos'], 0)} campos y {filas_2['registro']} filas en el registro.")
assert (filas_1["documentos"], filas_1["campos"], filas_1["registro"]) == (ESPERADO["archivos"] - 1, ESPERADO["campos"], ESPERADO["archivos"]), "C7: filas inesperadas en la base."
assert SEGUNDA.saltados == ESPERADO["archivos"] and SEGUNDA.resultados == [], "C7: la segunda corrida no debe procesar ningún archivo."
assert all(filas_2[tabla] == filas_1[tabla] == parquet_1[tabla] == parquet_2[tabla] for tabla in ("documentos", "campos", "registro")), "C7: las filas deben ser las mismas tras repetir."
CONTROLES.append("C7")
print("✔ C7 superado: DuckDB y Parquet con las mismas filas tras repetir")

Archivos: 200 | ya procesados: 200 | por procesar: 0
Control: 0 aceptados | 0 apartados (cuarentena o duplicado)
Listo en 0.2 s | 0.0 documentos por minuto | escritos por la cola: 0


tabla,filas en DuckDB,filas en Parquet,DuckDB tras repetir,Parquet tras repetir
documentos,199,199,199,199
campos,2.059,2.059,2.059,2.059
registro,200,200,200,200


estado,archivos
cuarentena,5
duplicado,1
ok,194


En esta corrida: la segunda corrida saltó 200 archivos y procesó 0, en 0,2 s. La base conserva 199 documentos, 2.059 campos y 200 filas en el registro.

✔ C7 superado: DuckDB y Parquet con las mismas filas tras repetir


**Cómo leer esta salida**

- **Qué es cada fila:** Una tabla de la base, con sus filas en DuckDB y en Parquet, antes y después de repetir la corrida.
- **Qué mirar:** Las cuatro columnas de números son iguales en cada fila, y la segunda corrida saltó todos los archivos.
- **Qué se decide:** Que la segunda corrida no procese nada es un acierto: el registro evita trabajo y filas repetidas.

In [20]:
# E4: la exactitud por fase y por tipo, la lista «revisar» y el reporte HTML
from mi_pipeline.almacenamiento.reporte import escribir_reporte, tabla_exactitud, tablas_del_reporte

campos_e4 = leer_tabla(CFG_4.almacenamiento.duckdb, "campos")
CORRECTOS = {Path(archivo).stem: correcto_de(Path(archivo).stem) for archivo in campos_e4["archivo"].unique()}
POR_FASE, POR_TIPO = tabla_exactitud(campos_e4, CORRECTOS, es_correcto)
ver(POR_FASE, "Exactitud después de cada etapa (toda la bandeja)")
ver(POR_TIPO, "Campos correctos / esperados por tipo, después de cada etapa")
revisar_e4 = tablas_del_reporte(CFG_4.almacenamiento.duckdb)["revisar"]
ver(revisar_e4.head(12), f"Lista «revisar»: los 12 primeros de {len(revisar_e4)} grupos de campos vacíos")
REPORTE = escribir_reporte(CFG_4.almacenamiento.duckdb, CFG_4.almacenamiento.reporte, exactitud=(POR_FASE, POR_TIPO), titulo="Reporte de la bandeja de la Mesa de Partes DEMO")
vacios = int(campos_e4["revisar"].sum())
decir(f"la exactitud sube en cada etapa: {' -> '.join(pct(valor) for valor in POR_FASE['exactitud'])}. Quedan {num(vacios, 0)} campos vacíos de {num(len(campos_e4), 0)} "
      f"para revisión humana. El reporte quedó en {REPORTE.as_posix()}: se abre con doble clic.")
assert list(POR_FASE["correctos"]) == sorted(POR_FASE["correctos"]), "La exactitud no debe bajar de una etapa a la siguiente."

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\curso\\datos\\10_Ground_Truth\\LOT-0001.json'

**Cómo leer esta salida**

- **Qué es cada fila:** En la primera tabla, una etapa con los campos correctos hasta ahí; en la segunda, un tipo de documento; en la tercera, un campo que quedó vacío y en cuántos documentos.
- **Qué mirar:** La exactitud sube en cada etapa, y hay tipos que quedan muy abajo.
- **Qué se decide:** Qué tipos de documento necesitan revisión humana completa y cuáles pueden pasar con una muestra.

In [ ]:
# E4: la línea de comandos. El mismo pipeline, sin cuaderno: python -m mi_pipeline (C8)
CORTA = armar_bandeja("corta")                       # la lista corta de 60 archivos: la corrida tarda menos
salida_cli = (SALIDAS / "cli").resolve()
shutil.rmtree(salida_cli, ignore_errors=True)
comando = [sys.executable, "-m", "mi_pipeline", "--entrada", str(CORTA), "--salida", str(salida_cli), "--modo", "procesos", "--trabajadores", "4"]
print("En la terminal, desde la carpeta del clon, es este comando:")
print("python -m mi_pipeline --entrada ..\\" + CORTA.name + " --salida s06_pipeline\\salidas\\cli --modo procesos --trabajadores 4\n")
primera_cli = subprocess.run(comando, cwd=RAIZ, capture_output=True, text=True, encoding="utf-8", errors="replace")
print(primera_cli.stdout[-900:] or primera_cli.stderr[-900:])
segunda_cli = subprocess.run(comando, cwd=RAIZ, capture_output=True, text=True, encoding="utf-8", errors="replace")
print("Otra vez el mismo comando:\n" + "\n".join(segunda_cli.stdout.strip().splitlines()[-3:]))
assert primera_cli.returncode == 0 and segunda_cli.returncode == 0, "C8: la línea de comandos debe terminar sin error."
assert (salida_cli / "reporte.html").is_file() and (salida_cli / "parquet" / "campos.parquet").is_file(), "C8: faltan el reporte o los Parquet."
assert "60 saltados" in segunda_cli.stdout, "C8: la segunda corrida del comando no debe procesar nada."
CONTROLES.append("C8")
print("✔ C8 superado: la línea de comandos corre de punta a punta y deja el reporte en salidas/cli/reporte.html")

En la terminal, desde la carpeta del clon, es este comando:
python -m mi_pipeline --entrada ..\bandeja_s06_corta --salida s06_pipeline\salidas\cli --modo procesos --trabajadores 4



**Cómo leer esta salida**

- **Qué se ve:** Lo que imprime el comando la primera vez y las últimas líneas de la segunda.
- **Qué mirar:** La primera vez procesa los 60 archivos; la segunda dice «60 saltados».
- **Qué se decide:** El comando y su YAML son lo que se entrega para que otra persona repita el proceso.

In [ ]:
# Cierre: los controles superados, las pruebas de mi_pipeline y el tiempo del cuaderno
todo_pasa = correr_pruebas()
faltan = [control for control in ("C0", "C0r", "C1", "C2", "C3", "C4", "C5", "C6", "C7", "C8") if control not in CONTROLES]
print("Controles superados:", ", ".join(CONTROLES), "| faltan:", ", ".join(faltan) or "ninguno")
print("Minutos del cuaderno:", num((time.perf_counter() - INICIO) / 60, 1))
assert todo_pasa and not faltan, "Falta algún control o alguna prueba no pasa: se revisa el paso correspondiente de la guía."

## Reto extra

El participante que termina antes cambia `trabajadores` a 3 en una copia de la configuración (`CFG.con(trabajadores=3)`), corre la bandeja corta y compara los documentos por minuto con los de 2 y 4 procesos.